In [1]:
from pathlib import Path
import os, json, hashlib, shutil, time, zipfile
import urllib.request

os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "2")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from sklearn.model_selection import train_test_split

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "app/api.py").is_file()),
    None
)
assert PROJECT_ROOT is not None, "Save the notebook inside your project."

CLASSES = [
    "cardboard", "glass", "metal",
    "paper", "plastic", "trash"
]

keras.utils.set_random_seed(42)

print("Project:", PROJECT_ROOT)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Classes:", CLASSES)

I0000 00:00:1795278894.883154   25700 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Project: /workspaces/dl_final
TensorFlow: 2.21.0
Keras: 3.15.1
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [2]:
RAW = PROJECT_ROOT / "raw_data"
RAW.mkdir(exist_ok=True)
DATA = RAW / "dataset-resized"
archive = RAW / "dataset-resized.zip"

url = (
    "https://raw.githubusercontent.com/garythung/trashnet/"
    "6fa2b878c6c1b4304b91109070ce0edf9279bb31/"
    "data/dataset-resized.zip"
)

if not DATA.exists():
    if not archive.exists():
        print("Downloading TrashNet...")
        urllib.request.urlretrieve(url, archive)

    expected = "0bf472790f8b20e5c950d5b5012a9d38af0d3392efd65f8ce171334fc16b07c2"
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected

    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            destination = (RAW / item.filename).resolve()
            assert destination.is_relative_to(RAW.resolve())
            if "__MACOSX" not in item.filename and not Path(item.filename).name.startswith("._"):
                z.extract(item, RAW)

files = pd.DataFrame([
    {"cls": c, "file": p.name, "path": str(p)}
    for c in CLASSES
    for p in sorted((DATA / c).glob("*.jpg"))
])

assert len(files) == 2527, "Dataset incomplete."

files["md5"] = [
    hashlib.md5(Path(p).read_bytes()).hexdigest()
    for p in files["path"]
]

bad = files.duplicated("md5", keep=False)
print("Duplicate images removed:", int(bad.sum()))
clean = files.loc[~bad].copy()

train, rest = train_test_split(
    clean, test_size=0.30, stratify=clean["cls"], random_state=42
)
val, test = train_test_split(
    rest, test_size=0.50, stratify=rest["cls"], random_state=42
)

split = pd.concat([
    train.assign(split="train"),
    val.assign(split="val"),
    test.assign(split="test")
])

def load_image(path):
    image = tf.io.decode_image(
        tf.io.read_file(path), channels=3, expand_animations=False
    )
    return tf.image.resize(image, [160, 160])

def make_ds(frame, training=False):
    paths = frame["path"].tolist()
    labels = frame["cls"].map(CLASSES.index).to_numpy()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    options = tf.data.Options()
    options.threading.private_threadpool_size = 2
    options.threading.max_intra_op_parallelism = 1
    ds = ds.with_options(options)

    if training:
        ds = ds.shuffle(len(frame), seed=42)

    ds = ds.map(
        lambda p, y: (load_image(p), y),
        num_parallel_calls=2
    )
    return ds.batch(32).prefetch(1), paths, labels

train_ds, _, y_train = make_ds(train, True)
val_ds, val_paths, y_val = make_ds(val)
test_ds, _, y_test = make_ds(test)

print("Train / validation / test:", len(train), len(val), len(test))
majority = np.bincount(y_train).argmax()
print("Baseline accuracy:", round(float((y_val == majority).mean()), 4))

Duplicate images removed: 6
Train / validation / test: 1764 378 379
Baseline accuracy: 0.2354


In [3]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

base = keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet"
)
base.trainable = False

model = keras.Sequential([
    keras.Input((160, 160, 3)),
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.1),
    keras.layers.Rescaling(1 / 127.5, offset=-1, name="rescaling"),
    base,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(6, activation="softmax", name="head")
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("STAGE 1: train the head")
h1 = model.fit(
    train_ds, validation_data=val_ds, epochs=20,
    callbacks=[
        keras.callbacks.EarlyStopping(
            patience=5, restore_best_weights=True
        )
    ],
    shuffle=False, verbose=2
)

frozen_weights = model.get_weights()
frozen_loss = min(h1.history["val_loss"])

print("Choice rule: fine-tuning must improve validation loss by >= 0.01")

base.trainable = True
start = [layer.name for layer in base.layers].index("block_13_expand")

for layer in base.layers[:start]:
    layer.trainable = False

for layer in base.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("STAGE 2: fine-tuning")
h2 = model.fit(
    train_ds, validation_data=val_ds, epochs=15,
    callbacks=[
        keras.callbacks.EarlyStopping(
            patience=4, restore_best_weights=True
        )
    ],
    shuffle=False, verbose=2
)

gain = frozen_loss - min(h2.history["val_loss"])
chosen = "fine-tuned" if gain >= 0.01 else "frozen"

if chosen == "frozen":
    model.set_weights(frozen_weights)

print("CHOSEN:", chosen, "| validation-loss gain:", round(gain, 4))

plt.plot(
    h1.history["val_loss"] + h2.history["val_loss"],
    label="validation loss"
)
plt.axvline(len(h1.history["val_loss"]) - 0.5, ls="--")
plt.legend()
plt.show()

pv = model.predict(val_ds, verbose=0)
print("VALIDATION")
print(classification_report(
    y_val, pv.argmax(1),
    labels=list(range(6)), target_names=CLASSES, zero_division=0
))

threshold = 0.99
target_met = False

for t in np.round(np.arange(0.20, 0.991, 0.01), 2):
    keep = pv.max(1) >= t
    if keep.any() and (pv.argmax(1)[keep] == y_val[keep]).mean() >= 0.95:
        threshold = float(t)
        target_met = True
        break

print("Threshold:", threshold, "| validation target met:", target_met)

# Final test: no model/threshold changes after this.
pt = model.predict(test_ds, verbose=0)
test_accuracy = accuracy_score(y_test, pt.argmax(1))
test_f1 = f1_score(y_test, pt.argmax(1), average="macro")

print("TEST accuracy:", round(test_accuracy, 4))
print("TEST macro-F1:", round(test_f1, 4))
print("TRAINING COMPLETE")

STAGE 1: train the head
Epoch 1/20


W0000 00:00:1795278909.309834   25775 cpu_allocator_impl.cc:82] Allocation of 78643200 exceeds 10% of free system memory.
W0000 00:00:1795278909.406942   25775 cpu_allocator_impl.cc:82] Allocation of 80621568 exceeds 10% of free system memory.
W0000 00:00:1795278910.406395   25775 cpu_allocator_impl.cc:82] Allocation of 78643200 exceeds 10% of free system memory.
W0000 00:00:1795278910.486625   25775 cpu_allocator_impl.cc:82] Allocation of 80621568 exceeds 10% of free system memory.
W0000 00:00:1795278911.202816   25775 cpu_allocator_impl.cc:82] Allocation of 78643200 exceeds 10% of free system memory.


56/56 - 48s - 863ms/step - accuracy: 0.5153 - loss: 1.2873 - val_accuracy: 0.6931 - val_loss: 0.8417
Epoch 2/20
56/56 - 43s - 766ms/step - accuracy: 0.7251 - loss: 0.7545 - val_accuracy: 0.7540 - val_loss: 0.6890
Epoch 3/20
56/56 - 44s - 791ms/step - accuracy: 0.7562 - loss: 0.6464 - val_accuracy: 0.7831 - val_loss: 0.6216
Epoch 4/20
56/56 - 70s - 1s/step - accuracy: 0.7749 - loss: 0.5946 - val_accuracy: 0.8095 - val_loss: 0.5891
Epoch 5/20
56/56 - 55s - 974ms/step - accuracy: 0.7880 - loss: 0.5432 - val_accuracy: 0.8095 - val_loss: 0.5487
Epoch 6/20
56/56 - 56s - 993ms/step - accuracy: 0.8163 - loss: 0.4950 - val_accuracy: 0.8228 - val_loss: 0.5383
Epoch 7/20
56/56 - 37s - 652ms/step - accuracy: 0.8339 - loss: 0.4635 - val_accuracy: 0.8201 - val_loss: 0.5537
Epoch 8/20
56/56 - 51s - 904ms/step - accuracy: 0.8424 - loss: 0.4288 - val_accuracy: 0.8201 - val_loss: 0.5459
Epoch 9/20
56/56 - 66s - 1s/step - accuracy: 0.8469 - loss: 0.4327 - val_accuracy: 0.8280 - val_loss: 0.5314
Epoch 10/

KeyboardInterrupt: 

In [ ]:
MODEL_DIR = PROJECT_ROOT / "models/trash"
DATA_DIR = PROJECT_ROOT / "data/trash"
SAMPLES = DATA_DIR / "samples"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES.mkdir(parents=True, exist_ok=True)

keras.Model(model.inputs, model.outputs).save(
    MODEL_DIR / "trash_tl.keras"
)

meta = {
    "title": "Waste sorting (TrashNet)",
    "classes": CLASSES,
    "classes_bn": [
        "কার্ডবোর্ড", "কাচ", "ধাতু",
        "কাগজ", "প্লাস্টিক", "অন্যান্য আবর্জনা"
    ],
    "image_size": 160,
    "model_file": "trash_tl.keras",
    "model_name": f"MobileNetV2 {chosen}",
    "not_sure_threshold": threshold,
    "threshold_target_met_on_validation": target_met,
    "gradcam_layer": "out_relu",
    "gradcam_tolerance": 1e-4,
    "test_looks": 1
}

(MODEL_DIR / "meta.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

split[["file", "cls", "split"]].to_csv(
    DATA_DIR / "trash_split.csv", index=False
)

def gradcam(m, x):
    backbone = next(layer for layer in m.layers if isinstance(layer, keras.Model))
    head = m.get_layer("head")

    with tf.GradientTape() as tape:
        maps = backbone(m.get_layer("rescaling")(x), training=False)
        tape.watch(maps)
        scores = tf.reduce_mean(maps, axis=(1, 2)) @ head.kernel + head.bias
        top = int(tf.argmax(scores[0]))
        selected = scores[:, top]

    grads = tape.gradient(selected, maps)
    weights = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.nn.relu(
        tf.reduce_sum(maps * weights[:, None, None, :], axis=-1)
    )[0].numpy()

    return tf.nn.softmax(scores)[0].numpy(), cam / (cam.max() + 1e-9)

known = {}

for k, cls in enumerate(CLASSES):
    for i in np.where(y_val == k)[0][:2]:
        path = Path(val_paths[i])
        name = f"{cls}__{path.name}"
        shutil.copy2(path, SAMPLES / name)

        p, cam = gradcam(model, load_image(str(path))[None])
        known[name] = {
            "probabilities": p.round(6).tolist(),
            "cam": cam.round(6).tolist()
        }

(DATA_DIR / "known_answers.json").write_text(
    json.dumps(known), encoding="utf-8"
)

saved = keras.models.load_model(
    MODEL_DIR / "trash_tl.keras", compile=False
)

for name, expected in known.items():
    x = load_image(str(SAMPLES / name))[None]
    np.testing.assert_allclose(
        saved.predict(x, verbose=0)[0],
        expected["probabilities"],
        atol=1e-4, rtol=0
    )

print("MODEL SAVED:", MODEL_DIR)
print("APP DATA SAVED:", DATA_DIR)
print("Samples:", len(known))
print("SAVED MODEL VERIFIED")

## Live class completion
Training stopped early to finish the app. The following executed cell retains the learned weights, evaluates, exports and verifies them. Original notebook is preserved.

In [ ]:

from pathlib import Path
import json, shutil
import numpy as np
import tensorflow as tf
import keras
from sklearn.metrics import accuracy_score, f1_score

assert "model" in globals(), "No trained model exists in this kernel."
assert "val_ds" in globals() and "val_paths" in globals(), "Dataset variables are missing."
lr = float(keras.ops.convert_to_numpy(model.optimizer.learning_rate))
chosen = "fine-tuned (stopped early)" if lr < 0.0001 else "head-trained (stopped early)"
print("Keeping existing learned weights:", chosen)
pv = model.predict(val_ds, verbose=0)
threshold, target_met = 0.99, False
for t in np.round(np.arange(0.20, 0.991, 0.01), 2):
    keep = pv.max(1) >= t
    if keep.any() and (pv.argmax(1)[keep] == y_val[keep]).mean() >= 0.95:
        threshold, target_met = float(t), True
        break
pt = model.predict(test_ds, verbose=0)
test_accuracy = float(accuracy_score(y_test, pt.argmax(1)))
test_f1 = float(f1_score(y_test, pt.argmax(1), average="macro"))
print("Test accuracy:", round(test_accuracy, 4), "macro-F1:", round(test_f1, 4))
print("Threshold:", threshold, "validation target met:", target_met)
MODEL_DIR = PROJECT_ROOT / "models/trash"
DATA_DIR = PROJECT_ROOT / "data/trash"
SAMPLES = DATA_DIR / "samples"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES.mkdir(parents=True, exist_ok=True)
keras.Model(model.inputs, model.outputs).save(MODEL_DIR / "trash_tl.keras")
meta = {
    "title": "Waste sorting (TrashNet)", "classes": CLASSES,
    "classes_bn": ["কার্ডবোর্ড", "কাচ", "ধাতু", "কাগজ", "প্লাস্টিক", "অন্যান্য আবর্জনা"],
    "image_size": 160, "model_file": "trash_tl.keras", "model_name": "MobileNetV2 " + chosen,
    "not_sure_threshold": threshold, "threshold_target_met_on_validation": target_met,
    "gradcam_layer": "out_relu", "gradcam_tolerance": 1e-4, "test_looks": 1,
    "training_note": "Existing full-data training interrupted for the live class; weights retained, no new fit call."
}
(MODEL_DIR / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
split[["file", "cls", "split"]].to_csv(DATA_DIR / "trash_split.csv", index=False)
def completion_cam(m, x):
    backbone = next(l for l in m.layers if isinstance(l, keras.Model))
    head = m.get_layer("head")
    with tf.GradientTape() as tape:
        maps = backbone(m.get_layer("rescaling")(x), training=False)
        tape.watch(maps)
        scores = tf.reduce_mean(maps, axis=(1, 2)) @ head.kernel + head.bias
        selected = scores[:, int(tf.argmax(scores[0]))]
    grads = tape.gradient(selected, maps)
    weights = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.nn.relu(tf.reduce_sum(maps * weights[:, None, None, :], axis=-1))[0].numpy()
    return tf.nn.softmax(scores)[0].numpy(), cam / (cam.max() + 1e-9)
known = {}
for k, cls in enumerate(CLASSES):
    for i in np.where(y_val == k)[0][:2]:
        path = Path(val_paths[i])
        name = cls + "__" + path.name
        shutil.copy2(path, SAMPLES / name)
        p, cam = completion_cam(model, load_image(str(path))[None])
        known[name] = {"probabilities": p.round(6).tolist(), "cam": cam.round(6).tolist()}
(DATA_DIR / "known_answers.json").write_text(json.dumps(known), encoding="utf-8")
(DATA_DIR / "run_report.json").write_text(json.dumps({"accuracy": test_accuracy, "macro_f1": test_f1, "training": chosen}, indent=2), encoding="utf-8")
saved = keras.models.load_model(MODEL_DIR / "trash_tl.keras", compile=False)
for name, expected in known.items():
    x = load_image(str(SAMPLES / name))[None]
    np.testing.assert_allclose(saved.predict(x, verbose=0)[0], expected["probabilities"], atol=1e-4, rtol=0)
print("SAVED MODEL VERIFIED:", MODEL_DIR, "| samples:", len(known))


Keeping existing learned weights: head-trained (stopped early)


Test accuracy: 0.8391 macro-F1: 0.8095
Threshold: 0.89 validation target met: True


SAVED MODEL VERIFIED: /workspaces/dl_final/models/trash | samples: 12
